# Cross-Attention Fusion Training Demo (No ROC Plot)

This notebook mimics the training workflow of `roc_curve_demo.ipynb` but removes plotting and replaces late-fusion with a cross-attentive fusion module inspired by the Global-Local Graph paper.

Main differences from late fusion:
- Build two aligned branches: sequence/global branch and PSSM branch.
- Use symmetric cross-attentive gating between branches before classification.
- Optionally add HSIC regularization to encourage complementary features.

In [1]:
import os
os.environ['PYTHONHASHSEED'] = '22'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import json
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    accuracy_score, f1_score, matthews_corrcoef, brier_score_loss
)
import tensorflow as tf
from tensorflow import keras

np.random.seed(22)
tf.random.set_seed(22)

from proteinbert import (
    load_anticrispr_with_ids,
    load_pretrained_model,
    load_feature_cache,
    attach_pssm_features,
)
from proteinbert.conv_and_global_attention_model import get_model_with_hidden_layers_as_outputs
from proteinbert.pssm_fusion import _encode_x

PROJECT_ROOT = '/home/nemophila/projects/protein_bert'
BENCHMARKS_DIR = os.path.join(PROJECT_ROOT, 'anticrispr_benchmarks')
WORK_ROOT = os.environ.get('PSSM_WORK_ROOT', '/home/nemophila/data/pssm_work')
FEAT_DIR = os.path.join(WORK_ROOT, 'features')
PSSM_DIM = '1110'
SEED = 22

parquet_path = os.path.join(FEAT_DIR, f'pssm_features_{PSSM_DIM}.parquet')
csv_path = os.path.join(FEAT_DIR, f'pssm_features_{PSSM_DIM}.csv')
cache_path = parquet_path if os.path.exists(parquet_path) else csv_path
if not os.path.exists(cache_path):
    raise FileNotFoundError(f'PSSM cache not found: {cache_path}')

print('Using PSSM cache:', cache_path)

train_base, test_base = load_anticrispr_with_ids(BENCHMARKS_DIR, benchmark_name='anticrispr_binary')
feat_df, feat_cols = load_feature_cache(cache_path)
train_df = attach_pssm_features(train_base, feat_df, feat_cols)
test_df = attach_pssm_features(test_base, feat_df, feat_cols)

sub_train, sub_valid = train_test_split(
    train_df, test_size=0.1, stratify=train_df['label'], random_state=SEED
)

print('train:', sub_train.shape, 'valid:', sub_valid.shape, 'test:', test_df.shape)
print('Acr counts (train/valid/test):', sub_train['label'].sum(), sub_valid['label'].sum(), test_df['label'].sum())

2026-03-20 20:09:41.931821: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0


Using PSSM cache: /home/nemophila/data/pssm_work/features/pssm_features_1110.csv
train: (996, 1113) valid: (111, 1113) test: (286, 1113)
Acr counts (train/valid/test): 184 21 26


In [2]:
@dataclass
class CrossAttTrainConfig:
    seq_len: int = 512
    batch_size: int = 8
    frozen_epochs: int = 6
    unfrozen_epochs: int = 12
    frozen_lr: float = 1e-4
    unfrozen_lr: float = 2e-5
    patience: int = 4
    global_dropout: float = 0.3
    pssm_dropout: float = 0.3
    global_bottleneck_dim: int = 64
    global_hidden_dim: int = 128
    pssm_hidden_dim: int = 128
    cross_att_dim: int = 128
    fusion_hidden_dim: int = 128
    hsic_lambda: float = 1e-3

cfg = CrossAttTrainConfig()
print(cfg)

def find_best_threshold(y_true, y_prob, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr = 0.5
    best_f1 = -1.0
    for thr in grid:
        cur = f1_score(y_true, (y_prob >= thr).astype(int))
        if cur > best_f1:
            best_f1 = cur
            best_thr = float(thr)
    return best_thr

def evaluate_binary_full(y_true, y_prob, threshold=0.5):
    y_cls = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_cls).ravel()
    return {
        'AUC': float(roc_auc_score(y_true, y_prob)),
        'AUPRC': float(average_precision_score(y_true, y_prob)),
        'F1': float(f1_score(y_true, y_cls)),
        'MCC': float(matthews_corrcoef(y_true, y_cls)),
        'Brier': float(brier_score_loss(y_true, y_prob)),
        'ACC': float(accuracy_score(y_true, y_cls)),
        'SN': float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0,
        'SP': float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0,
        'Threshold': float(threshold),
    }

def empirical_hsic(x, y):
    x = x - tf.reduce_mean(x, axis=0, keepdims=True)
    y = y - tf.reduce_mean(y, axis=0, keepdims=True)
    k = tf.matmul(x, x, transpose_b=True)
    r = tf.matmul(y, y, transpose_b=True)
    n = tf.shape(x)[0]
    n_f = tf.cast(n, tf.float32)

    # Build centering matrix with dynamic shape to stay Keras-graph safe.
    eye_n = tf.eye(n, dtype=tf.float32)
    ones_n = tf.ones_like(eye_n)
    h = eye_n - ones_n / tf.maximum(n_f, 1.0)

    kh = tf.matmul(k, h)
    rh = tf.matmul(r, h)
    denom = tf.maximum((n_f - 1.0) * (n_f - 1.0), 1.0)
    return tf.linalg.trace(tf.matmul(kh, rh)) / denom

CrossAttTrainConfig(seq_len=512, batch_size=8, frozen_epochs=6, unfrozen_epochs=12, frozen_lr=0.0001, unfrozen_lr=2e-05, patience=4, global_dropout=0.3, pssm_dropout=0.3, global_bottleneck_dim=64, global_hidden_dim=128, pssm_hidden_dim=128, cross_att_dim=128, fusion_hidden_dim=128, hsic_lambda=0.001)


In [3]:
class SymmetricCrossAttention(keras.layers.Layer):
    def __init__(self, att_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att_dim = att_dim
        self.dropout = keras.layers.Dropout(dropout)
        self.q_seq = keras.layers.Dense(att_dim, use_bias=False)
        self.k_seq = keras.layers.Dense(att_dim, use_bias=False)
        self.v_seq = keras.layers.Dense(att_dim, use_bias=False)
        self.q_pssm = keras.layers.Dense(att_dim, use_bias=False)
        self.k_pssm = keras.layers.Dense(att_dim, use_bias=False)
        self.v_pssm = keras.layers.Dense(att_dim, use_bias=False)

    def call(self, inputs, training=None):
        seq_feat, pssm_feat = inputs

        q_s = self.q_seq(seq_feat)
        k_s = self.k_seq(seq_feat)
        v_s = self.v_seq(seq_feat)

        q_p = self.q_pssm(pssm_feat)
        k_p = self.k_pssm(pssm_feat)
        v_p = self.v_pssm(pssm_feat)

        scale = tf.sqrt(tf.cast(self.att_dim, tf.float32))
        alpha_s_from_p = tf.sigmoid(tf.reduce_sum(q_s * k_p, axis=-1, keepdims=True) / scale)
        alpha_p_from_s = tf.sigmoid(tf.reduce_sum(q_p * k_s, axis=-1, keepdims=True) / scale)

        cross = 0.5 * (alpha_s_from_p * v_p + alpha_p_from_s * v_s)
        cross = self.dropout(cross, training=training)
        return cross

def build_cross_attention_fusion_model(pretrained_model_generator, seq_len, pssm_dim, freeze_pretrained_layers, cfg):
    base_model = pretrained_model_generator.create_model(seq_len, compile=False, init_weights=True)
    base_model = get_model_with_hidden_layers_as_outputs(base_model)

    if freeze_pretrained_layers:
        for layer in base_model.layers:
            layer.trainable = False

    _, global_output = base_model.output
    pssm_input = keras.layers.Input(shape=(pssm_dim,), name='pssm_input')

    global_branch = keras.layers.LayerNormalization(name='global_ln_in')(global_output)
    global_branch = keras.layers.Dense(cfg.global_bottleneck_dim, activation='relu', name='global_bottleneck')(global_branch)
    global_branch = keras.layers.Dropout(cfg.global_dropout, name='global_drop')(global_branch)
    global_branch = keras.layers.Dense(cfg.global_hidden_dim, activation='relu', name='global_dense')(global_branch)
    global_branch = keras.layers.LayerNormalization(name='global_ln_out')(global_branch)

    pssm_branch = keras.layers.LayerNormalization(name='pssm_ln_in')(pssm_input)
    pssm_branch = keras.layers.Dense(cfg.pssm_hidden_dim, activation='relu', name='pssm_dense')(pssm_branch)
    pssm_branch = keras.layers.Dropout(cfg.pssm_dropout, name='pssm_drop')(pssm_branch)
    pssm_branch = keras.layers.LayerNormalization(name='pssm_ln_out')(pssm_branch)

    seq_aligned = keras.layers.Dense(cfg.cross_att_dim, activation='linear', name='seq_align')(global_branch)
    pssm_aligned = keras.layers.Dense(cfg.cross_att_dim, activation='linear', name='pssm_align')(pssm_branch)

    cross_feat = SymmetricCrossAttention(cfg.cross_att_dim, dropout=cfg.pssm_dropout, name='cross_attention')([seq_aligned, pssm_aligned])

    fused = keras.layers.Concatenate(name='cross_fusion_concat')([seq_aligned, pssm_aligned, cross_feat])
    fused = keras.layers.Dense(cfg.fusion_hidden_dim, activation='relu', name='fusion_dense')(fused)
    fused = keras.layers.Dropout(cfg.pssm_dropout, name='fusion_drop')(fused)
    out = keras.layers.Dense(1, activation='sigmoid', name='output')(fused)

    model = keras.models.Model(inputs=base_model.inputs + [pssm_input], outputs=out)

    if cfg.hsic_lambda > 0:
        hsic_term = empirical_hsic(seq_aligned, pssm_aligned)
        model.add_loss(cfg.hsic_lambda * hsic_term)

    return model

In [4]:
pmg, enc = load_pretrained_model(
    local_model_dump_dir=os.path.join(PROJECT_ROOT, 'proteinbert_models'),
    download_model_dump_if_not_exists=True,
    validate_downloading=False,
)

x_tr = sub_train[feat_cols].to_numpy(dtype=np.float32)
x_va = sub_valid[feat_cols].to_numpy(dtype=np.float32)
x_te = test_df[feat_cols].to_numpy(dtype=np.float32)

scaler = StandardScaler()
x_tr_s = scaler.fit_transform(x_tr)
x_va_s = scaler.transform(x_va)
x_te_s = scaler.transform(x_te)

y_tr = sub_train['label'].astype(int).to_numpy()
y_va = sub_valid['label'].astype(int).to_numpy()
y_te = test_df['label'].astype(int).to_numpy()

X_tr = _encode_x(enc, sub_train['seq'].tolist(), cfg.seq_len, x_tr_s)
X_va = _encode_x(enc, sub_valid['seq'].tolist(), cfg.seq_len, x_va_s)
X_te = _encode_x(enc, test_df['seq'].tolist(), cfg.seq_len, x_te_s)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=cfg.patience, restore_best_weights=True)
]

model = build_cross_attention_fusion_model(
    pmg,
    seq_len=cfg.seq_len,
    pssm_dim=len(feat_cols),
    freeze_pretrained_layers=True,
    cfg=cfg,
)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=cfg.frozen_lr),
    loss='binary_crossentropy',
)
print('Stage 1: train with frozen pretrained layers')
model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=cfg.frozen_epochs,
    batch_size=cfg.batch_size,
    callbacks=callbacks,
    verbose=0,
)

for layer in model.layers:
    layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=cfg.unfrozen_lr),
    loss='binary_crossentropy',
)
print('Stage 2: train with unfrozen layers')
model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=cfg.unfrozen_epochs,
    batch_size=cfg.batch_size,
    callbacks=callbacks,
    verbose=0,
)

2026-03-20 20:09:44.134859: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2026-03-20 20:09:44.136286: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2026-03-20 20:09:44.144770: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:2a:00.0 name: NVIDIA L40S computeCapability: 8.9
coreClock: 2.52GHz coreCount: 142 deviceMemorySize: 44.53GiB deviceMemoryBandwidth: 804.75GiB/s
2026-03-20 20:09:44.144817: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
2026-03-20 20:09:44.147760: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.11
2026-03-20 20:09:44.147905: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublasLt.so.11
2026-03-2

Stage 1: train with frozen pretrained layers


2026-03-20 20:09:47.308405: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:116] None of the MLIR optimization passes are enabled (registered 2)
2026-03-20 20:09:47.309004: I tensorflow/core/platform/profile_utils/cpu_utils.cc:112] CPU Frequency: 2500000000 Hz
2026-03-20 20:09:54.163554: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.11
2026-03-20 20:09:54.943835: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublasLt.so.11
2026-03-20 20:09:54.959680: I tensorflow/stream_executor/cuda/cuda_blas.cc:1838] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-03-20 20:09:54.960732: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudnn.so.8
2026-03-20 20:09:57.830814: W tensorflow/stream_executor/gpu/asm_compiler.cc:63] Running ptxas --version returned 256
2026-03

Stage 2: train with unfrozen layers


In [5]:
COMPARISON_RESULTS_DIR = '/home/nemophila/projects/protein_bert/Comparison/results'

y_prob_va = model.predict(X_va, batch_size=cfg.batch_size, verbose=0).reshape(-1)
best_thr = find_best_threshold(y_va, y_prob_va)
y_prob_te = model.predict(X_te, batch_size=cfg.batch_size, verbose=0).reshape(-1)
metrics = evaluate_binary_full(y_te, y_prob_te, threshold=best_thr)

print(f'Cross-attention fusion test AUC={metrics["AUC"]:.3f}, AUPRC={metrics["AUPRC"]:.3f}, best_thr={best_thr:.2f}')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

os.makedirs(COMPARISON_RESULTS_DIR, exist_ok=True)
np.savez(
    os.path.join(COMPARISON_RESULTS_DIR, 'cross_attention_predictions.npz'),
    y_true=y_te,
    y_prob=y_prob_te,
)
with open(os.path.join(COMPARISON_RESULTS_DIR, 'cross_attention_metrics.json'), 'w') as f:
    json.dump({'method': 'ProteinBERT+PSSM1110 CrossAttention', 'metrics': metrics}, f, indent=2)

print('Saved:', os.path.join(COMPARISON_RESULTS_DIR, 'cross_attention_predictions.npz'))
print('Saved:', os.path.join(COMPARISON_RESULTS_DIR, 'cross_attention_metrics.json'))

Cross-attention fusion test AUC=0.919, AUPRC=0.635, best_thr=0.40
  AUC: 0.9188
  AUPRC: 0.6349
  F1: 0.5667
  MCC: 0.5227
  Brier: 0.0658
  ACC: 0.9091
  SN: 0.6538
  SP: 0.9346
  Threshold: 0.4000
Saved: /home/nemophila/projects/protein_bert/Comparison/results/cross_attention_predictions.npz
Saved: /home/nemophila/projects/protein_bert/Comparison/results/cross_attention_metrics.json
